<img src="logo.png" alt="Vegeta" width="240">

# NISUS-Zero — photographing birds up close: the mission software prototype (32)

Nisus-Zero (notebook 31) carries a Jetson and a camera. Its mission here: find the birds flying over an area, track
several at once, pick them **one by one**, fly close enough for a sharp close-up of each, and come home on the
energy the autopilot budgets. The detector will be **YOLOX**, but it is **not run** in this prototype: its detections
are simulated from the truth, with the latency and the rates of an **ASSUMED** Jetson compute budget (Part 2), until
the TensorRT engine exists and is benchmarked on the hardware.

The mission software is a separate, general package, **`vegeta.mission`**: typed messages on a small in-process bus
with a deterministic executor (no ROS 2 for now; the nodes and messages map one to one onto ROS 2 later). It never
imports a simulator. The simulator side is `vegeta.mission.sim` (birds, the simulated detector, a kinematic aircraft
for fast loops) and, for the 6-DOF aircraft, `designs/nisus_birds.py`.

| part | what | where the code is |
|---|---|---|
| 1 | architecture: nodes, topics, the seam to the autopilot | `vegeta.mission` (`pipeline`, `bus`, `messages`) |
| 2 | YOLOX on the Jetson: the assumed compute budget | `vegeta.mission.compute` |
| 3 | camera and photo geometry: lens, pixels on the bird, detection range, blur | `vegeta.mission.camera` |
| 4 | algorithms: which tracker, filter, selection and guidance, and why | `tracking`, `targeting`, `guidance`, `photo` |
| 5 | one encounter with the kinematic aircraft: tracks against the truth | `vegeta.mission.sim` |
| 6 | trade study (kinematic, several seeds): lens, Jetson, model | `vegeta.mission.sim` |
| 7 | the whole missions in MuJoCo (Chiron 6-DOF) with videos | `designs/nisus_birds.py`, `nisus_birds_movie.py` |
| 8 | what the prototype shows, and what to do next | — |

Switches (environment variables): `NISUS_QUICK=1` short hunts, two missions, one seed; `NISUS_MOVIES=0` no videos
(pyvista needs a display: run under `xvfb-run -a` on a headless machine).

In [ ]:
import math, os, re, sys, time, json
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Video, display, Markdown
sys.path.insert(0, "designs")
import vegeta.mission as vm
from vegeta.mission import compute, camera, pipeline, metrics
from vegeta.mission.sim import BirdField, BirdGroup, SPECIES, run_kinematic
from vegeta.mission.guidance import GuidanceConfig
from vegeta.mission.photo import PhotoConfig
import nisus_birds as nb

pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 220)
QUICK = os.environ.get("NISUS_QUICK") == "1"
MOVIES = os.environ.get("NISUS_MOVIES", "1") != "0"
OUT = Path("output") / "32_nisus_birds"; OUT.mkdir(parents=True, exist_ok=True)
SEEDS = (1,) if QUICK else (1, 2, 3)
STATUS = {}
print(f"QUICK {QUICK} | movies {'on' if MOVIES else 'off'} | seeds {SEEDS}")
def save(df, name):
    df.to_csv(OUT / f"{name}.csv")
    return df

# Part 1 — architecture

```
autopilot (Nisus flight controller)          companion computer (Jetson): vegeta.mission
   nav state  ───────────────────────────►  nav ─┬─► detector ──► detections ──► tracker ──► tracks
   camera frames ─────────────────────────► camera                                  │
                                                 │      ▲ target, tracks (ROI)        ▼
   course, height, airspeed  ◄──── guidance ◄──── target ◄──── scheduler ◄── tracks, photos
                                                 └─► photo trigger ──► photos ──► scheduler
                                                     compute monitor: detections ──► health
```

- **Messages** (`messages.py`): frozen dataclasses with a time stamp: `NavState`, `CameraFrame`, `Detections`,
  `Tracks`, `TargetSelection`, `GuidanceCmd`, `PhotoEvent`, `Health`. Frames: world ENU, body x forward / y left /
  z up, camera OpenCV.
- **Bus and executor** (`bus.py`): named topics (latest value and per-subscriber queues, everything recorded for
  replay) and nodes run at their own rates on the caller's clock: repeatable to the bit, no threads, no ROS 2.
- **Nodes**: `DetectorNode` (when to grab a frame, which crop, the latency; `SimDetector` now, `YoloxTrtDetector`
  later), `BirdTracker`, `TargetScheduler`, `PhotoPassGuidance`, `PhotoTrigger`, `ComputeMonitor`.
- **The seam to the autopilot**: the autopilot publishes its navigation state and the camera pose; it flies the
  `GuidanceCmd` (course over the ground, height, airspeed) in its mode 'birds', keeps launch, landing, the energy
  manager and the failsafes to itself (no fresh command for 5 s: hold, then return).

In [ ]:
# the flight software imports no simulator (the test suite checks the same)
pkg = Path(vm.__file__).parent
rows = []
for f in sorted(pkg.glob("*.py")):
    src = f.read_text()
    imports = sorted(set(re.findall(r"^\s*(?:from|import)\s+([\w.]+)", src, re.M)))
    sim = [m for m in imports if any(k in m for k in ("mujoco", "chiron", "rclpy", "sim", "nisus"))]
    rows.append({"module": f.name, "imports": ", ".join(imports), "simulator imports": ", ".join(sim) or "none"})
display(pd.DataFrame(rows).set_index("module"))
cfg0 = nb.BirdScenario().config()
display(pd.Series(cfg0.rates, name="node rates [Hz]").to_frame())

# Part 2 — YOLOX on the Jetson: the compute budget (ASSUMED)

latency = CPU overhead + GFLOPs / effective throughput; GPU load = Σ rate × GPU time per pass.

- **Published**: the YOLOX models' size and GFLOPs (Ge et al. 2021); the Jetson peaks (Nano 472 GFLOPS FP16; Orin
  Nano Super 67 TOPS INT8 sparse, so ~33 dense INT8 and ~17 FP16).
- **Assumed**: the effective throughput a small detector gets (a few % to ~20 % of the peak) and the overheads. They put
  YOLOX-Tiny at ~12 fps (416 px) on the Nano and ~100+ fps on the Orin Nano Super: the order of community TensorRT
  reports for similar detectors, **not a measurement of this pipeline**.
- **Pipeline**: birds are small, so the full-frame **search** pass cuts the 3280 × 2464 frame into 2 × 2 tiles of
  640 px (5 Hz), and while a target is selected a **crop** of 832 px around it goes through at 416 px (15 Hz).
  The Nano B01 cannot carry that: its configuration in the missions is YOLOX-Nano, and the budget scales its rates
  down where it is over (`search Hz achieved`).

In [ ]:
B = compute.budget_table()
cols = ["device", "model", "precision", "search input", "GFLOPs search", "latency search [ms]", "latency roi [ms]", "GPU load", "feasible",
        "search [Hz] achieved", "roi [Hz] achieved", "module power [W]", "status"]
display(save(B[cols].round(2), "compute_budget"))
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.barh(B.index, 100 * B["GPU load"].astype(float), color=["#d62728" if not f else "#2ca02c" for f in B["feasible"]])
ax.axvline(80, color="k", ls="--", lw=1); ax.text(81, 0, "80 % cap", fontsize=8)
ax.set_xlabel("GPU load at 5 Hz search + 15 Hz crop [%] (ASSUMED)"); ax.set_xscale("log"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "compute_budget.png", dpi=120); plt.show()
ref = compute.budget(cfg0.detector)
STATUS["compute"] = f"{ref['device']} + {ref['model']}: GPU load {100 * ref['GPU load']:.0f} % (ASSUMED, not benchmarked)"
print(STATUS["compute"])

# Part 3 — camera and photo geometry

The vision camera is the IMX219 module (3280 × 2464, 1.12 µm). With its stock 3.04 mm lens it sees 62° × 49°: wide
for finding birds, but a bird is small in it. A 6 mm M12 lens (34° × 26°, a few grams, ~€10, *assumed* part) doubles the
pixels on the bird at every range: twice the detection range and twice the distance at which a close-up is possible.
A "good" photo here: **≥ 150 px across** the wings (the full frame), **≤ 2 px motion blur**, the bird within 75 % of
the half-diagonal from the centre. The birds' **flight distances** (they flee a small drone at roughly 18-35 m:
assumed, after Vas et al. 2015 and Mulero-Pázmány et al. 2017) set how close one can get before the bird escapes.

In [ ]:
rows = []
for lname, cam in nb.LENSES.items():
    for sp in ("black-headed gull", "carrion crow", "feral pigeon"):
        s = SPECIES[sp]
        px = lambda r: cam.pixels_across(s.wingspan_m, r) * 0.8          # wings at a mean attitude
        net_search = 2 * 640 / cam.width                                  # network pixels per image pixel (2 x 2 tiles)
        net_roi = 416 / 832
        rows.append({"lens": lname, "HFOV [deg]": round(cam.hfov_deg, 1), "species": sp, "wingspan [m]": s.wingspan_m, "flees at [m]": s.fear_m,
                     "px at 20 m": round(px(20)), "px at 50 m": round(px(50)), "good photo (150 px) within [m]": round(cam.fx * s.wingspan_m * 0.8 / 150, 1),
                     "50 % detection, search [m]": round(cam.fx * s.wingspan_m * 0.8 * net_search / 7.0), "50 % detection, crop [m]": round(cam.fx * s.wingspan_m * 0.8 * net_roi / 7.0),
                     "blur at 5 m/s crossing, 25 m [px]": round(cam.blur_px(np.array([5.0, 0, 0]), np.array([0, 25.0, 0]), np.zeros(3), None), 2)})
G3 = save(pd.DataFrame(rows), "camera_geometry"); display(G3)
fig, ax = plt.subplots(figsize=(9, 4))
r = np.linspace(5, 120, 200)
for lname, cam in nb.LENSES.items():
    ax.plot(r, cam.pixels_across(1.0, r) * 0.8, label=f"gull, {lname}")
ax.axhline(150, color="k", ls="--", lw=1); ax.text(100, 155, "good: 150 px", fontsize=8)
ax.axvspan(0, SPECIES["black-headed gull"].fear_m, color="orange", alpha=0.15, label="gull flees (assumed)")
ax.set_xlabel("range [m]"); ax.set_ylabel("pixels across the wings"); ax.set_yscale("log"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig(OUT / "pixels_vs_range.png", dpi=120); plt.show()

# Part 4 — the algorithms: options and choices

**Detection → tracks (multi-object tracking).**

| option | idea | for birds from a small aircraft |
|---|---|---|
| SORT (Bewley 2016) | Kalman filter per object + Hungarian matching on IoU | IoU is ~0 for small, fast birds between frames: matching fails |
| DeepSORT / BoT-SORT | + appearance re-identification (a ReID network), camera-motion compensation | birds of a species look alike; a second network costs Jetson time; deferred |
| **ByteTrack** (Zhang 2022) | match high-score boxes first, then low-score boxes to the leftover tracks | **chosen**: small, blurred, far birds give low scores; they keep tracks alive without starting tracks from clutter |
| **OC-SORT** (Cao 2023) | observation-centric re-update after a gap; robust to non-linear motion | **its re-update is used** to recover a bird after a turn or a gap |
| JPDA / MHT | probabilistic or multi-hypothesis association | the right tools for dense flocks; heavier; the next step if flocks matter |
| GM-PHD / random finite sets | no explicit association | good for counting a murmuration, not for following one bird |

**State per bird.** A monocular camera measures the bearing precisely, the range only from the bird's apparent size
(an assumed wingspan; ±40 % in log range). So the filter works in **world coordinates** (the aircraft's own motion
is removed with its navigation state: no image registration needed) with an **EKF in the camera's coordinates**
(normalised image position + ln range) and an **IMM** (interacting multiple models) over constant velocity and a
coordinated turn (a gull circling in a thermal). Two lessons from building it, kept in the code: fusing the detection
as a 3-D point with a long covariance along the ray made tracks collapse onto the aircraft (every ray passes through
the camera); and the size-based range errors are correlated (species, wing attitude), so the range uncertainty needs a
floor or the filter mispredicts the parallax of the aircraft's motion.

**One bird at a time.** Cost = straight-line intercept time at the dash speed + the turn to it + the track's
uncertainty; commit to the cheapest (hysteresis, no switch in the first 20 s nor within 150 m), release it after two
good photos (done), on a timeout or when its track is lost (blacklisted for 30 s). The alternatives for a longer
mission are an orienteering / travelling-salesman plan over the tracks, or a value per species.

**Getting close with a fixed camera.** A fixed-wing aircraft cannot hover or orbit a bird and keep it in the frame of a
nose camera: it must point at it. Head-on passes close at 25-30 m/s (one second in range, fast line-of-sight rotation,
blur); **stern passes** close at a few m/s. The guidance: intercept towards a point 25 m behind the bird (predicted
along its arc when it circles), then pursue it at its speed plus a closure that shrinks near 18 m, the camera's 15°
look-down compensated by flying above it; break off (90° away, climb 10 m) when any recent track is predicted inside
10 m (with a 0.65 range margin for the uncertain range); reposition and come again. A **gimbal** would allow a standoff
orbit (more photos, less disturbance): the main hardware option to study next.

**The shutter.** Fires from the target's last box (measured pixels, independent of the uncertain range) when the bird
spans ≥ 0.55 × 150 px; the photo counts as good by the trigger's estimate; the simulator judges the same photo from
the truth (the numbers reported are the truth's).

# Part 5 — one encounter with the kinematic aircraft

The whole mission stack against simulated birds (four black-headed gulls circling in a thermal, two carrion crows
wandering) with a kinematic aircraft that follows the guidance within its bank, roll-rate and climb limits: the same
code the MuJoCo missions run, in seconds. The scores (CLEAR-MOT and IDF1 against the truth, the photos judged from
the truth) are computed on the birds the camera could see.

In [ ]:
def field(seed, centre=(0.0, 300.0)):
    c = np.asarray(centre)
    return BirdField([BirdGroup("black-headed gull", 4, tuple(c), "thermal", (50, 85), 35),
                      BirdGroup("carrion crow", 2, tuple(c + [120, 80]), "wander", (40, 70), 120)], seed=seed)

def kin_cfg(lens="6 mm M12", device="orin_nano_super", model="yolox-tiny"):
    return nb.mission_config(device, model, lens, field_centre=(0.0, 300.0))

t0 = time.time()
loop = run_kinematic(field(1), kin_cfg(), duration=120 if QUICK else 300, seed=1)
print(f"{time.time() - t0:.1f} s wall")
display(pd.Series(loop.scores(), name="encounter (seed 1)").to_frame())
ev = pd.DataFrame(loop.stack.guidance.events, columns=["t", "target", "transition"]); display(ev.head(25))
fig, ax = plt.subplots(figsize=(8, 8))
nav = np.array([r["nav_pos"] for r in loop.rec]); ax.plot(nav[:, 0], nav[:, 1], color="#1f77b4", lw=1, label="aircraft")
B_ = np.array([r["birds"] for r in loop.rec])
for i in range(B_.shape[1]):
    ax.plot(B_[:, i, 0], B_[:, i, 1], color="k", lw=0.6, alpha=0.6)
tr = {}
for r in loop.rec:
    for tid, st, pos, cov, box in r["tracks"]:
        if st == "confirmed":
            tr.setdefault(tid, []).append(pos)
for tid, ps in tr.items():
    ps = np.array(ps); ax.plot(ps[:, 0], ps[:, 1], ".", ms=2)
for p in loop.photos:
    j = min(range(len(loop.rec)), key=lambda k: abs(loop.rec[k]["t"] - p["t"]))
    ax.plot(*loop.rec[j]["nav_pos"][:2], "*", color="gold" if p["good_true"] else "grey", ms=10 if p["good_true"] else 5)
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_title("truth (black), confirmed tracks (dots), photos (stars; gold: good)"); ax.legend()
plt.tight_layout(); plt.savefig(OUT / "encounter_top_view.png", dpi=120); plt.show()
display(pd.DataFrame(loop.photos).round(2).head(20))

# Part 6 — trade study (kinematic loop, several seeds)

The lens, the Jetson and the model, each on the same bird fields (different seeds: different birds' starts and
thermals). Means over the seeds; the spread is large with a handful of birds, so read the trends, not the decimals.

In [ ]:
configs = {"Orin Nano Super, YOLOX-Tiny, 6 mm": ("6 mm M12", "orin_nano_super", "yolox-tiny"),
           "Orin Nano Super, YOLOX-Tiny, stock 3 mm": ("stock 3.04 mm", "orin_nano_super", "yolox-tiny"),
           "Orin Nano Super, YOLOX-S INT8, 6 mm": ("6 mm M12", "orin_nano_super", "yolox-s"),
           "Nano B01, YOLOX-Nano, 6 mm": ("6 mm M12", "nano_b01", "yolox-nano")}
if QUICK:
    configs = dict(list(configs.items())[:2])
rows = {}
for name, (lens, dev, model) in configs.items():
    res = []
    for seed in SEEDS:
        lp = run_kinematic(field(seed), kin_cfg(lens, dev, model), duration=120 if QUICK else 300, seed=seed)
        s = lp.scores(); s["photos >= 120 px (truth)"] = sum(p["px_true"] >= 120 for p in lp.photos)
        res.append(s)
    df = pd.DataFrame(res)
    rows[name] = df[["good photos (truth)", "birds photographed well", "photos", "photos >= 120 px (truth)", "MOTA", "IDF1", "recall",
                     "identity switches", "closest approach to any bird [m]", "approaches under 10 m", "breakoffs", "detector frames"]].mean()
T6 = save(pd.DataFrame(rows).T.round(2), "trade_kinematic"); display(T6)

# Part 7 — the whole missions in MuJoCo (Chiron 6-DOF)

Nisus-Zero of notebook 31 (the same Chiron model, coefficient table, propulsion map, battery and energy manager): hand
launch, climb, outbound to the bird area 350 m away, **six minutes of hunting** under `vegeta.mission` in the flight
controller's mode 'birds', the return, the approach and the landing. The scenarios: the reference (Orin Nano Super,
YOLOX-Tiny, 6 mm lens; gulls in a thermal and crows), the stock lens, the Nano B01 with YOLOX-Nano, a pigeon flock
(fast birds) with two herring gulls, a 5 m/s wind with turbulence, and the Jetson failing during the hunt (the
autopilot's failsafe brings the aircraft home). The videos are rendered from the log (synthetic camera view: see
Part 1 of the video's caption).

In [ ]:
scns = nb.bird_scenarios()
if QUICK:
    scns = [replace(s, bird_time_s=120.0) for s in scns[:1]] + [replace(scns[2], bird_time_s=120.0)]
EPS = []
for bs in scns:
    t0 = time.time()
    ep = nb.run(bs)
    EPS.append(ep)
    print(f"{bs.name} / {bs.device} / {bs.lens}: {time.time() - t0:.0f} s wall, landed {ep.outcome.get('success')}, "
          f"{len(ep.birds.loop.photos)} photos")
CMP = save(nb.compare(EPS), "missions"); display(CMP)

In [ ]:
for ep in EPS:
    display(Markdown(f"**{ep.bird_scenario.label}**"))
    pt = nb.photo_table(ep)
    if len(pt):
        display(pt.sort_values("px_true", ascending=False).head(8))
    near = pd.DataFrame(ep.birds.loop.near)
    if len(near):
        display(near.round(1))
    pt.to_csv(OUT / f"{ep.bird_scenario.slug}_photos.csv", index=False)

In [ ]:
# the videos (pyvista: under xvfb-run on a headless machine)
VIDEOS = {}
if MOVIES:
    import nisus_birds_movie as nm
    for ep in EPS:
        t0 = time.time()
        p = nm.render_movie(ep, OUT / f"{ep.bird_scenario.slug}.mp4")
        VIDEOS[ep.bird_scenario.name + " / " + ep.bird_scenario.device + " / " + ep.bird_scenario.lens] = p
        print(f"{p}  ({time.time() - t0:.0f} s)")
    first = next(iter(VIDEOS.values()))
    display(Video(str(first), embed=False, width=960))
else:
    print("movies: NOT RUN (NISUS_MOVIES=0)")

# Part 8 — what the prototype shows, and what next

Read the tables above for the numbers of your run. The development run of the six missions (2026-10-09, seed 1, six
minutes of hunting each) gave:

| mission | photos | good (truth) | birds photographed well | best [px] | closest approach [m] | landed |
|---|---|---|---|---|---|---|
| reference: Orin Nano Super, YOLOX-Tiny, 6 mm | 36 | 2 | 2 | 171 | 8.0 | yes |
| stock 3.04 mm lens | 19 | 0 | 0 | 98 | 12.6 | yes |
| Nano B01, YOLOX-Nano | 18 | 0 | 0 | 154 | 9.2 | yes |
| pigeon flock | 30 | 8 | 2 | 196 | **4.7** | yes |
| wind 5 m/s | 36 | 0 | 0 | 134 | 14.5 | yes |
| Jetson failure at 200 s | 5 | 0 | 0 | 147 | 11.6 | yes (failsafe return) |

What that shows (one seed each: trends, not statistics):

- **The pipeline fits the Orin Nano Super** with room to spare (about a third of the GPU in the assumed budget for
  YOLOX-Tiny with 2 × 2 tiled search and a target crop). The **Nano B01** cannot run YOLOX-Tiny at these rates; YOLOX-Nano
  fits at ~80 % but detects small birds later (assumed). Benchmark before believing either.
- **Lens**: the stock 3 mm lens never produced a 150 px photo (best 98 px): a close-up would need half the range, inside
  the birds' flight distance. The 6 mm lens is the recommended first change (a few grams, a few euros).
- **Close-ups are scarce**: a fixed nose camera, birds that flee at 18-35 m and a minimum separation of 10 m leave a
  window of a few metres in range; most photos are 80-140 px, a few cross 150 px. Stern passes keep the blur under a
  pixel at 1/4000 s. In wind the stern geometry is harder (the bird's ground track drifts): no good photo in that run.
  The ways to more close-ups: a gimbal (standoff orbits, the bird kept in the frame through turns), a narrower photo
  camera beside the wide vision camera, a quieter approach (gliding, motor at idle), species with longer tolerance.
- **Tracking**: recall on the birds in view 0.46-0.68; MOTA and IDF1 are low because identities fragment: the narrow
  forward camera loses each bird for long stretches (behind, beside, in a turn) and the size-based range is coarse.
  Appearance re-identification is the obvious addition once real images exist.
- **Safety: not yet good enough in flocks.** In the pigeon flock the aircraft came within 4.7 m of a pigeon (two
  approaches under 10 m): fast birds crossing from outside the camera's view cannot be seen, let alone avoided, and
  the 10 m separation is enforced only on tracks. Before any flight near birds: a wider separation in flocks, no passes
  into a flock's path, and the rules of 5n.7. The birds' escape behaviour is an assumption that drives these numbers.
- **Failsafe**: when the Jetson died at 200 s the autopilot held for 5 s without guidance, then returned and landed.

Next (todo Stage 5n): run YOLOX on recorded flight video and fit the detector model (`px50`, the score spread, the
false positives); benchmark the TensorRT engines on the Orin Nano Super; fly the pipeline on the bench with replayed
video (hardware in the loop); a gimbal study; an appearance re-ID; flock scenarios with JPDA.

In [ ]:
STATUS["missions"] = f"{len(EPS)} MuJoCo missions, {sum(len(e.birds.loop.photos) for e in EPS)} photos, " \
                     f"{sum(sum(p['good_true'] for p in e.birds.loop.photos) for e in EPS)} good (truth)"
STATUS["YOLOX"] = "NOT RUN (simulated detections; the TensorRT engine and its benchmark are todo 5n)"
STATUS["videos"] = ", ".join(str(p) for p in VIDEOS.values()) if VIDEOS else "NOT RUN"
display(pd.Series(STATUS, name="status").to_frame())